In [2]:
from pathlib import Path
import pandas as pd

raw_dir = Path("C:/Users/visva/Desktop/Folder/Document/GUVI/Projects/P1/data/raw")

files = {
    "customers": "olist_customers_dataset.csv",
    "geolocation": "olist_geolocation_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "payments": "olist_order_payments_dataset.csv",
    "reviews": "olist_order_reviews_dataset.csv",
    "orders": "olist_orders_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv"
}

dataset={}

for table_name, file_name in files.items():
    dataset[table_name]=pd.read_csv(raw_dir/file_name)

print ("Loaded Tables", list(dataset.keys()))

Loaded Tables ['customers', 'geolocation', 'order_items', 'payments', 'reviews', 'orders', 'products', 'sellers', 'category_translation']


In [3]:
cleaned_data = {
    table_name: df.copy()
    for table_name, df in dataset.items()
}

In [4]:
orders = cleaned_data["orders"]

order_date_columns = [
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date"
]

for column in order_date_columns:
    report = (
        orders
        .assign(is_missing=orders[column].isnull())
        .groupby("order_status")["is_missing"]
        .agg(
            total_orders="size",
            missing_count="sum"
        )
    )

    report["missing_percentage"] = (
        report["missing_count"] /
        report["total_orders"] * 100
    ).round(2)

    print(f"\nMissing report: {column}")
    display(report[report["missing_count"] > 0])
    


Missing report: order_approved_at


,total_orders,missing_count,missing_percentage
order_status,,,
canceled,625,141,22.56
created,5,5,100.00
delivered,96478,14,0.01



Missing report: order_delivered_carrier_date


,total_orders,missing_count,missing_percentage
order_status,,,
approved,2,2,100.0
canceled,625,550,88.0
created,5,5,100.0
delivered,96478,2,0.0
invoiced,314,314,100.0
processing,301,301,100.0
unavailable,609,609,100.0



Missing report: order_delivered_customer_date


,total_orders,missing_count,missing_percentage
order_status,,,
approved,2,2,100.00
canceled,625,619,99.04
created,5,5,100.00
delivered,96478,8,0.01
invoiced,314,314,100.00
processing,301,301,100.00
shipped,1107,1107,100.00
unavailable,609,609,100.00


In [5]:
products = cleaned_data["products"]
order_items = cleaned_data["order_items"]

products_with_missing_values = products[
    products.isnull().any(axis=1)
]

print(
    "Products containing missing values:",
    len(products_with_missing_values)
)

display(products_with_missing_values.head(10))

Products containing missing values: 611


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
244,e10758160da97891c2fdcbc35f0f031d,NaN,NaN,NaN,NaN,2200.0,16.0,2.0,11.0
294,39e3b9b12cd0bf8ee681bbc1c130feb5,NaN,NaN,NaN,NaN,300.0,16.0,7.0,11.0
299,794de06c32a626a5692ff50e4985d36f,NaN,NaN,NaN,NaN,300.0,18.0,8.0,14.0
347,7af3e2da474486a3519b0cba9dea8ad9,NaN,NaN,NaN,NaN,200.0,22.0,14.0,14.0
428,629beb8e7317703dcc5f35b5463fd20e,NaN,NaN,NaN,NaN,1400.0,25.0,25.0,25.0


In [6]:
missing_product_ids = products_with_missing_values["product_id"]

items_using_missing_products = order_items[
    order_items["product_id"].isin(missing_product_ids)
]

print(
    "Order items using products with missing metadata:",
    len(items_using_missing_products)
)

Order items using products with missing metadata: 1604


In [7]:
reviews = cleaned_data["reviews"]

review_report = reviews.groupby("review_score").agg(
    total_reviews=("review_id", "size"),

    missing_titles=(
        "review_comment_title",
        lambda column: column.isnull().sum()
    ),

    missing_messages=(
        "review_comment_message",
        lambda column: column.isnull().sum()
    )
)

review_report

,total_reviews,missing_titles,missing_messages
review_score,,,
1,11424,9551,2679
2,3151,2673,1006
3,8179,7355,4622
4,19142,17407,13166
5,57328,50670,36774


In [8]:
relationships = [
    ("orders", "customer_id", "customers", "customer_id"),
    ("order_items", "order_id", "orders", "order_id"),
    ("order_items", "product_id", "products", "product_id"),
    ("order_items", "seller_id", "sellers", "seller_id"),
    ("payments", "order_id", "orders", "order_id"),
    ("reviews", "order_id", "orders", "order_id")
]

for child_table, child_column, parent_table, parent_column in relationships:
    child_df = cleaned_data[child_table]
    parent_df = cleaned_data[parent_table]

    invalid_foreign_keys = (
        child_df[child_column].notnull()
        & ~child_df[child_column].isin(parent_df[parent_column])
    ).sum()

    print(
        f"{child_table}.{child_column} -> "
        f"{parent_table}.{parent_column}: "
        f"{invalid_foreign_keys} invalid values"
    )

orders.customer_id -> customers.customer_id: 0 invalid values
order_items.order_id -> orders.order_id: 0 invalid values
order_items.product_id -> products.product_id: 0 invalid values
order_items.seller_id -> sellers.seller_id: 0 invalid values
payments.order_id -> orders.order_id: 0 invalid values
reviews.order_id -> orders.order_id: 0 invalid values


In [9]:
geolocation = cleaned_data["geolocation"]

rows_before = len(geolocation)

cleaned_data["geolocation"] = (
    geolocation
    .drop_duplicates()
    .reset_index(drop=True)
)

rows_after = len(cleaned_data["geolocation"])

print("Rows before:", rows_before)
print("Rows after:", rows_after)
print("Removed:", rows_before - rows_after)

Rows before: 1000163
Rows after: 738332
Removed: 261831


In [10]:
cleaned_data["products"] = cleaned_data["products"].rename(
    columns={
        "product_name_lenght": "product_name_length",
        "product_description_lenght": "product_description_length"
    }
)


In [11]:
cleaned_data["products"]["product_category_name"] = (
    cleaned_data["products"]["product_category_name"]
    .fillna("unknown")
)

In [12]:
postal_code_columns = {
    "customers": "customer_zip_code_prefix",
    "sellers": "seller_zip_code_prefix",
    "geolocation": "geolocation_zip_code_prefix"
}

for table_name, column_name in postal_code_columns.items():
    cleaned_data[table_name][column_name] = (
        cleaned_data[table_name][column_name]
        .astype("string")
        .str.zfill(5)
    )

In [13]:
lowercase_columns = {
    "customers": ["customer_city"],
    "sellers": ["seller_city"],
    "geolocation": ["geolocation_city"],
    "orders": ["order_status"],
    "payments": ["payment_type"],
    "products": ["product_category_name"],
    "category_translation": [
        "product_category_name",
        "product_category_name_english"
    ]
}

for table_name, columns in lowercase_columns.items():
    for column in columns:
        cleaned_data[table_name][column] = (
            cleaned_data[table_name][column]
            .astype("string")
            .str.strip()
            .str.lower()
        )

In [14]:
state_columns = {
    "customers": "customer_state",
    "sellers": "seller_state",
    "geolocation": "geolocation_state"
}

for table_name, column in state_columns.items():
    cleaned_data[table_name][column] = (
        cleaned_data[table_name][column]
        .astype("string")
        .str.strip()
        .str.upper()
    )

In [15]:
date_columns = {
    "orders": [
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ],
    "order_items": [
        "shipping_limit_date"
    ],
    "reviews": [
        "review_creation_date",
        "review_answer_timestamp"
    ]
}

for table_name, columns in date_columns.items():
    for column in columns:
        missing_before = cleaned_data[table_name][column].isnull().sum()

        cleaned_data[table_name][column] = pd.to_datetime(
            cleaned_data[table_name][column],
            errors="coerce"
        )

        missing_after = cleaned_data[table_name][column].isnull().sum()

        print(
            f"{table_name}.{column}: "
            f"{missing_after - missing_before} invalid dates created"
        )

orders.order_purchase_timestamp: 0 invalid dates created
orders.order_approved_at: 0 invalid dates created
orders.order_delivered_carrier_date: 0 invalid dates created
orders.order_delivered_customer_date: 0 invalid dates created
orders.order_estimated_delivery_date: 0 invalid dates created
order_items.shipping_limit_date: 0 invalid dates created
reviews.review_creation_date: 0 invalid dates created
reviews.review_answer_timestamp: 0 invalid dates created


In [16]:
final_summary = []

for table_name, df in cleaned_data.items():
    final_summary.append({
        "table": table_name,
        "rows": len(df),
        "columns": len(df.columns),
        "missing_cells": df.isnull().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    })

final_summary_df = pd.DataFrame(final_summary)
final_summary_df

,table,rows,columns,missing_cells,duplicate_rows
0,customers,99441,5,0,0
1,geolocation,738332,5,0,0
2,order_items,112650,7,0,0
3,payments,103886,5,0,0
4,reviews,99224,7,145903,0
5,orders,99441,8,4908,0
6,products,32951,9,1838,0
7,sellers,3095,4,0,0
8,category_translation,71,2,0,0


In [17]:
cleaned_dir = raw_dir.parent / "cleaned"
cleaned_dir.mkdir(parents=True, exist_ok=True)

for table_name, file_name in files.items():
    output_path = cleaned_dir / file_name

    cleaned_data[table_name].to_csv(
        output_path,
        index=False,
        date_format="%Y-%m-%d %H:%M:%S"
    )

    print(f"Saved: {output_path}")

Saved: C:\Users\visva\Desktop\Folder\Document\GUVI\Projects\P1\data\cleaned\olist_customers_dataset.csv
Saved: C:\Users\visva\Desktop\Folder\Document\GUVI\Projects\P1\data\cleaned\olist_geolocation_dataset.csv
Saved: C:\Users\visva\Desktop\Folder\Document\GUVI\Projects\P1\data\cleaned\olist_order_items_dataset.csv
Saved: C:\Users\visva\Desktop\Folder\Document\GUVI\Projects\P1\data\cleaned\olist_order_payments_dataset.csv
Saved: C:\Users\visva\Desktop\Folder\Document\GUVI\Projects\P1\data\cleaned\olist_order_reviews_dataset.csv
Saved: C:\Users\visva\Desktop\Folder\Document\GUVI\Projects\P1\data\cleaned\olist_orders_dataset.csv
Saved: C:\Users\visva\Desktop\Folder\Document\GUVI\Projects\P1\data\cleaned\olist_products_dataset.csv
Saved: C:\Users\visva\Desktop\Folder\Document\GUVI\Projects\P1\data\cleaned\olist_sellers_dataset.csv
Saved: C:\Users\visva\Desktop\Folder\Document\GUVI\Projects\P1\data\cleaned\product_category_name_translation.csv
